# Kütüphaneler

In [283]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas.io.formats.excel as pife
import warnings
import pickle
import math
import os
from termcolor import colored
from pandas.errors import SettingWithCopyWarning
from scipy.stats import zscore

# Ayarlar

In [284]:
scope = 1
skip_last_month = False
# target_start_term  = '2022.09'
# target_end_term  = '2023.09'
target_start_term, target_end_term = None, None

os.system('color')
pd.options.display.float_format = '{:,.2f}'.format
pife.ExcelFormatter.header_style = None

warnings.filterwarnings('ignore', category = UserWarning)
warnings.filterwarnings('ignore', category = pd.errors.ParserWarning)
warnings.filterwarnings('ignore', category = SettingWithCopyWarning)

base_path = 'C:\\Users\\105617\\Desktop\\Workspace\\Panel\\'

pickle_folder_path = base_path + 'pickle' + '\\'
data_folder_path = base_path + 'data' + '\\'
output_folder_path = base_path + 'output' + '\\'
kr202_folder_path = data_folder_path + 'KR202' + '\\'
desnace_folder_path = data_folder_path + 'DES & NACE' + '\\'
ms_folder_path = data_folder_path + 'Müşteri Sınıfı' + '\\'
proje_folder_path = data_folder_path + 'Proje Kredileri' + '\\'
memzuc_folder_path = data_folder_path + 'Memzuç' + '\\'
yz_folder_path = data_folder_path + 'Yapay Zeka' + '\\'
eds_folder_path = data_folder_path + 'Entegre Derece Skor' + '\\'
fg_folder_path = data_folder_path + 'Finansal Güçlük' + '\\'
kroa_folder_path = data_folder_path + 'KRÖA' + '\\'
lgd_folder_path = data_folder_path + 'LGD' + '\\'
bcek_folder_path = data_folder_path + 'Bankamız Çek' + '\\'
dbcek_folder_path = data_folder_path + 'Diğer Banka Çek' + '\\'
ktcs_folder_path = data_folder_path + 'KTÇS' + '\\'

for p in [
    data_folder_path, output_folder_path, pickle_folder_path,
    kr202_folder_path, desnace_folder_path, ms_folder_path, proje_folder_path, memzuc_folder_path,
    yz_folder_path, fg_folder_path, kroa_folder_path, lgd_folder_path,
    bcek_folder_path, dbcek_folder_path, ktcs_folder_path
]:
    os.makedirs(p, exist_ok = True)
    
mali_file_name = 'Mali Tablolar.xlsx'
zombi_mali_file_name = 'Mali Tablolar - Zombi.xlsx'

proje_file_name = 'Proje Kredileri.csv'
kur_file_name = 'Kur Riski Çalışması.csv'
sektor_file_name = 'Sektör Risklilik Çalışması.csv'
nace_file_name = 'NACE Bilgileri.csv'

sube_file_name = 'Şube Bilgileri.xlsx'
mnvt_file_name = 'Müşteri No - VKN - TCKN.csv'

# Yardımcı Fonksiyonlar

## Tablo Manipülasyonu

In [285]:
def add_h_space(dfo, space=1):
    df = dfo.copy()
    w = df.shape[1]
    for _ in range(space):
        df.loc[len(df)] = [np.nan] * w
    return df

def add_v_space(dfo, space=1):
    return pd.concat([dfo, pd.DataFrame([[np.nan] * space], columns=[np.nan] * space)], axis=1)
    

def h_stack(dfl, space=3):
    df = dfl.copy()
    df = df[0].reset_index(drop=True)
    dfln = [df]
    dfna = pd.DataFrame({np.nan: [np.nan]})
    for dftt in dfl[1:]:
        dft = dftt.copy()
        for i in range(space):
            dfln.append(dfna)
        dfln.append(dft.reset_index(drop=True))
    df = pd.concat(dfln, axis=1).reset_index(drop=True)
    return df


def v_stack(dfl, space=3):
    df = dfl.copy()
    df = df[0].reset_index(drop=True)
    dfln = [df]
    dfna = pd.DataFrame(columns=df.columns)

    dfna = add_h_space(dfna, space)
    for dftt in dfl[1:]:
        dft = dftt.copy()
        l = df.shape[1]
        lt = dft.shape[1]
        diff = l - lt

        if diff > 0:
            dft = add_v_space(dft, diff)
        elif diff < 0:
            df = add_v_space(df, -diff)

        dfc = pd.DataFrame([dft.columns], columns=df.columns)
        dft.columns = df.columns
        dfln.append(dfna)
        dfln.append(dfc)
        dfln.append(dft.reset_index(drop=True))

    df = pd.concat(dfln).reset_index(drop=True)

    return df


def insert_header(df, col):
    header = [col] if type(col) is str else col
    return v_stack([pd.DataFrame(columns=header), df], 0)

## Verim Tabloları

In [286]:
def fix_verim_table(df, columns=None):
    start = 0
    if columns is None:
        start = 3
        columns = df.iloc[2, 1:]
    df = df.iloc[start:, 1:]
    df.columns = columns
    df.columns.name = None
    df = df.reset_index(drop=True)
    return df

def quick_export_verim_csv(data, output_file_name, sep=';', header=False, index=False, open_file=False):
    cl = [str(x) for x in range(0, 16)]
    df = pd.DataFrame()
    df['10'] = data
    for c in ['0', '15']:
        df[c] = 'x'
    for c in cl:
        if c not in ['0', '10', '15']:
            df[c] = np.nan
    df = df[cl]

    output_file_path = output_folder_path + output_file_name + '.csv'
    df.to_csv(output_file_path, sep=sep, header=header, index=index)

    if open_file:
        os.startfile(output_file_path)
    

## Hızlı Çıktı

In [287]:
def quick_export(df_export, output_file_name, sheet_name=None, open_file=True):
    output_file_path = output_folder_path + output_file_name + '.xlsx'
    writer = pd.ExcelWriter(output_file_path, engine = 'xlsxwriter')
    if type(df_export) is list:
        if sheet_name is None:
            sheet_name = [x + 1 for x in range(len(df_export))]
        for df, sheet in zip(df_export, sheet_name):
            df.to_excel(writer, sheet_name = sheet, index = False)
    else:
        if sheet_name is None:
            sheet_name = output_file_name
        df_export.to_excel(writer, sheet_name = sheet_name, index = False)
    writer.close()
    if open_file:
        os.startfile(output_file_path)


    # def quick_export_csv(df, file_name, open_file=False):
#     df.to_csv(output_folder_path + file_name  + '.csv', sep=';', encoding='ANSI', index=False)

## Pickle

In [288]:
def save_pickle(file, name, sub=''):
    os.makedirs(pickle_folder_path + sub, exist_ok = True)
    with open(pickle_folder_path + sub + name, 'wb') as handle:
        pickle.dump(file, handle, protocol=pickle.HIGHEST_PROTOCOL)

def load_pickle(name, sub=''):
    with open(pickle_folder_path + sub + name, 'rb') as handle:
        return pickle.load(handle)

## Toplu Dosya İşleme

In [289]:
def scan_folder(folder_path, extension='.xlsx', offset=0):
    file_list, date_list = [], []
    for file_name in os.listdir(folder_path):
        if file_name.endswith(extension) and not file_name.startswith('~$'):
            s = file_name.split('.')
            y = s[0][-4:]
            m = s[1]
            d = s[2][:2]
            file_list.append(file_name)
            date_list.append(y + '.' + m + '.' + d)

    sorted_list = sorted(zip(date_list, file_list))

    if target_end_term is not None:
        sorted_list = sorted_list[:[x[0][:7] for x in sorted_list].index(target_end_term) + 1 + offset]
    elif skip_last_month:
        sorted_list = sorted_list[:-1]

    if target_start_term is not None:
        sorted_list = sorted_list[[x[0][:7] for x in sorted_list].index(target_end_term) + offset:]
    else:
        sorted_list = sorted_list[-scope:]

    return file_list, date_list, sorted_list

In [290]:
def batch_import_files(sorted_list, data_folder_path, pickle_sub, pickle_name_base, fix_verim=False, csv=None, sheet_list=None):
    sorted_date_list, df_raw_list = [], []
    pickle_sub += '\\'
    pickle_name_base += '_'
    l = len(sorted_list)
    for i, (date, name) in enumerate(sorted_list):
        sorted_date_list.append(date[:7])
        pickle_name = pickle_name_base + date
        if os.path.exists(pickle_folder_path + pickle_sub + pickle_name):
            df = load_pickle(pickle_name, pickle_sub)
            ps = f'({round((i + 1) / l * 100)}%)'
            ps = ' ' * (len('(100%)') - len(ps)) + ps
            print(colored(name, 'light_green'), 'önbellekten okundu', colored(ps, 'light_green'))
        else:
            print(colored(name, 'light_green'), 'okunuyor')
            if csv is not None:
                sep, encoding, low_memory, index_col = ';', 'ANSI', False, True
                if type(csv) is list:
                    try:
                        sep = csv[0]
                        encoding = csv[1]
                        low_memory = csv[2]
                        index_col = csv[3]
                    except:
                        pass
                if index_col:
                    df = pd.read_csv(data_folder_path + name, sep=sep, encoding=encoding, low_memory=low_memory)
                else:
                    df = pd.read_csv(data_folder_path + name, sep=sep, encoding=encoding, low_memory=low_memory, index_col=index_col)

            else:
                xls = pd.ExcelFile(data_folder_path + name)
                df = pd.DataFrame()
                if sheet_list is None:
                    sheet_list = xls.sheet_names
                for sheet in sheet_list:
                    print('\t', colored(sheet, 'light_cyan'), 'okunuyor')
                    dfs = pd.read_excel(xls, sheet_name=sheet)
                    if fix_verim:
                        dfs = fix_verim_table(dfs)
                    df = pd.concat([df, dfs], ignore_index=True) if len(sheet_list) > 1 else dfs
                xls.close()
            print(colored(name, 'light_green'), 'kaydediliyor')
            save_pickle(df, pickle_name, pickle_sub)
            ps = f'({round((i + 1) / l * 100)}%)'
            ps = ' ' * (len('(100%)') - len(ps)) + ps
            print(colored(name, 'light_green'), 'kaydedildi', colored(ps, 'light_green'))

        df_raw_list.append(df.copy())
    print('Tüm dosyalar başarıyla okundu')

    return sorted_date_list, df_raw_list

In [291]:
def batch_process_files(df_raw_list, sorted_list, sorted_date_list, fix_df_function, column_list=None, fill_string=None, rename_exclude_columns = 'Müşteri No'):
    df_all = []
    df = pd.DataFrame()
    l = len(df_raw_list)
    for i, dfi in enumerate(df_raw_list):
        df = dfi.copy()
        
        if fix_df_function is not None:
            df = fix_df_function(df, i)

        if i == len(df_raw_list) - 1:
            df_last = df.copy().reset_index(drop=True)

        if column_list is not None:  
            df = df[column_list]
     
        if type(rename_exclude_columns) is str:
            df.columns = [c + ' ' + sorted_date_list[i] if c != rename_exclude_columns else c for c in df.columns]
        else:
            df.columns = [c + ' ' + sorted_date_list[i] if c not in rename_exclude_columns else c for c in df.columns]
        df = df.sort_values('Müşteri No').reset_index(drop=True)

        if fill_string is not None:
            df = df.fillna(fill_string)

        df_all.append(df)        
        ps = f'({round((i + 1) / l * 100)}%)'
        ps = ' ' * (len('(100%)') - len(ps)) + ps
        print(colored(sorted_list[i][1], 'light_green'), 'işlendi', colored(ps, 'light_green'))

    print('Tüm dosyalar başarıyla oluşturuldu')

    return df_all, df_last

# Tekil Dosyalar

## Mali Tablolar

In [292]:
if os.path.exists(pickle_folder_path + 'df_mali'):
    df_mali = load_pickle('df_mali')
else:
    xls = pd.ExcelFile(data_folder_path + mali_file_name)
    df_mali = pd.read_excel(xls)
    save_pickle(df_mali, 'df_mali')
    xls.close()
df_mali_backup = df_mali.copy()

## Zombi Firmalar Mali Tablolar

In [293]:
if os.path.exists(pickle_folder_path + 'df_zombi_mali'):
    df_zombi_mali = load_pickle('df_zombi_mali')
else:
    xls = pd.ExcelFile(data_folder_path + zombi_mali_file_name)
    df_zombi_mali = pd.read_excel(xls)
    save_pickle(df_zombi_mali, 'df_zombi_mali')
    xls.close()
df_zombi_mali_backup = df_zombi_mali.copy()

## Kur Riski Çalışması

In [294]:
if os.path.exists(pickle_folder_path + 'df_kur'):
    df_kur = load_pickle('df_kur')
else:
    df_kur = pd.read_csv(data_folder_path + kur_file_name, sep=';', encoding='utf-8')
    save_pickle(df_kur, 'df_kur')

## Sektör Risklilik Çalışması

In [295]:
if os.path.exists(pickle_folder_path + 'df_sektor'):
    df_sektor = load_pickle('df_sektor')
else:
    df_sektor = pd.read_csv(data_folder_path + sektor_file_name, sep=';', encoding='ANSI')
    save_pickle(df_sektor, 'df_sektor')

## Şube Bilgileri

In [296]:
if os.path.exists(pickle_folder_path + 'df_sube'):
    df_sube = load_pickle('df_sube')
else:
    xls = pd.ExcelFile(data_folder_path + sube_file_name)
    df_sube = pd.read_excel(xls)
    save_pickle(df_sube, 'df_sube')
    xls.close()

## NACE Bilgileri

In [297]:
if os.path.exists(pickle_folder_path + 'df_nace'):
    df_nace = load_pickle('df_nace')
else:
    df_nace = pd.read_csv(data_folder_path + nace_file_name, encoding='ANSI')
    for c in df_nace.columns:
        df_nace[c] = df_nace[c].apply(lambda x: x.strip())
    save_pickle(df_nace, 'df_nace')

## Müşteri No - VKN - TCKN

In [298]:
if os.path.exists(pickle_folder_path + 'df_mnvt'):
    df_mnvt = load_pickle('df_mnvt')
else:
    df_mnvt = pd.read_csv(data_folder_path + mnvt_file_name, sep=';')
    save_pickle(df_mnvt, 'df_mnvt')

df_mnvt = df_mnvt.loc[(df_mnvt['TCKN'].notnull()) | (df_mnvt['VKN'].notnull())]

# Veri İçe Alma

## KR202

In [299]:
kr202_file_list, kr202_date_list, kr202_sorted_list = scan_folder(kr202_folder_path)
kr202_sorted_date_list, kr202_df_raw_list = batch_import_files(kr202_sorted_list, kr202_folder_path, 'KR202', 'kr202')

last_date = kr202_sorted_list[-1][0][:7]

KR202 - 2023.10.31.xlsx önbellekten okundu (100%)
Tüm dosyalar başarıyla okundu


In [300]:
# # for df, date in zip(kr202_df_raw_list, kr202_sorted_date_list):
# #     cl = list(df.loc[df['Musteri No'].notnull(), 'Musteri No'])
# #     quick_export_verim_csv(cl, 'Müşteri Listesi - ' + date)

df = kr202_df_raw_list[-1]
date = '2023.11'
cl = list(df.loc[df['Musteri No'].notnull(), 'Musteri No'])
quick_export_verim_csv(cl, 'Müşteri Listesi - ' + date)

In [301]:
kr202_column_list = [
    'Müşteri No',
    'Şube Kodu',
    'İlgili Bölüm',
    'Yetki Kodu',
    'Tahsis Sektör Adı',
    'Nakdi TL Reeskontlu',
    'Nakdi YP Reeskontlu',
    'Nakdi Reeskontlu Toplam',
    'G.Nakdi TL Bakiye',
    'G.Nakdi YP Bakiye',
    'G.Nakdi Toplam',
    'Toplam Reeskontlu Risk',
]

kr202_include_list = [
    'Müşteri No', 'Adı', 'Şube Kodu', 'Şube Adı', 'Şube İl', 'Şube İlçe',
    'Şube Türü', 'Yetki Kodu', 'İlgili Bölüm', 'Risk Grubu Kodu', 'Risk Grubu Adı',
    'Tahsis Sektör Adı', 'Tahsis Alt Sektör Adı',
    'Nakdi TL Bakiye', 'Nakdi YP Bakiye', 'Nakdi Toplam',
    'G.Nakdi TL Bakiye', 'G.Nakdi YP Bakiye', 'G.Nakdi Toplam', 'Toplam Risk',
    'Nakdi TL Reeskontlu', 'Nakdi YP Reeskontlu', 'Nakdi Reeskontlu Toplam', 'Toplam Reeskontlu Risk'
]

kr202_column_rename_dict = {
    'Musteri No': 'Müşteri No',
    'Adi': 'Adı',
    'SANTRAL_SUBE_KODU': 'Şube Kodu',
    'SANTRAL SUBE KODU': 'Şube Kodu',
    'Santral Şube Kodu': 'Şube Kodu',
    'Santral Şube Adı': 'Şube Adı',
    'SANTRAL_SUBE_IL': 'Şube İl',
    'SANTRAL ŞUBE İL': 'Şube İl',
    'Santral  Şube İl': 'Şube İl',
    'SANTRAL_SUBE_ILCE': 'Şube İlçe',
    'SANTRAL ŞUBE İLÇE': 'Şube İlçe',
    'Santral Şube İlçe': 'Şube İlçe',
    'Santral Şube Türü': 'Şube Türü',
    'SEKTÖR': 'Tahsis Sektör Adı',
    'ALT SEKTÖR': 'Tahsis Alt Sektör Adı',
    'Tahsis Alt Sektör AdI': 'Tahsis Alt Sektör Adı',
}

kr202_kktb_list = [
    'KKTB',
    'Kurumsal Krediler Tahsis',
    'Kurumsal Kredi Krediler Tahsis',
    'Kurumsal Kredi Krediler Tahsis Bölümü'
]

kr202_tktb_list = [
    'TKTB',
    'Ticari Krediler Tahsis',
    'Ticari Kredi Krediler Tahsis',
    'Ticari Kredi Krediler Tahsis Bölümü'
]

kr202_tsa_rename_dict = {
    'İNŞAAT': 'İNŞAAT TAAHHÜT',
    'DİĞER HİZMET': 'DİĞER HİZMET FAALİYETLERİ',
    'ELEKTRONİK  BİLİŞİM  ': 'ELEKTRONİK BİLİŞİM',
    'ELEKTRONİK  BİLİŞİM': 'ELEKTRONİK BİLİŞİM',
    'ELEKTRONİK BİLİŞİM  ': 'ELEKTRONİK BİLİŞİM',
    'BELEDİYELER VE OSB LER': 'BELEDİYELER VE OSB\'LER',
    'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVM\'ler) ': 'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVM\'LER)',
    'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVM\'ler)': 'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVM\'LER)',
    'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVMler) ': 'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVM\'LER)',
    'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVMler)': 'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVM\'LER)',
}


def fix_kr202_bolum(df):
    x = df['İlgili Bölüm']
    if x in kr202_kktb_list:
        return 'Kurumsal Krediler Tahsis Bölümü'
    elif x in kr202_tktb_list:
        return 'Ticari Krediler Tahsis Bölümü'
    else:
        return x


def fix_kr202_df(df, i):
    for x in kr202_column_rename_dict:
        try:
            df = df.rename(columns={x: kr202_column_rename_dict[x]})
        except:
            pass
    
    if i == len(kr202_df_raw_list) - 1:
        df = df.loc[:, df.columns.isin(kr202_include_list)]
    else:
        df = df[kr202_column_list]

    df = df.loc[df['Müşteri No'].notnull()]
    df['İlgili Bölüm'] = df.apply(fix_kr202_bolum, axis=1)    
    df['Tahsis Sektör Adı'] = df['Tahsis Sektör Adı'].map(lambda x: kr202_tsa_rename_dict.get(x, x))

    return df

    
df_kr202_all, df_kr202_last = batch_process_files(kr202_df_raw_list, kr202_sorted_list, kr202_sorted_date_list, fix_kr202_df, kr202_column_list)


customer_list = []
for df in df_kr202_all:
    customer_list.append(list(df['Müşteri No']))

df_kr202_last_original_backup = df_kr202_last.copy()

KR202 - 2023.10.31.xlsx işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


# İSO500

In [240]:
# if os.path.exists(pickle_folder_path + 'df_gdlr'):
#     df_gdlr = load_pickle('df_gdlr')
# else:
#     fn = 'Grup_Firmalı_Limit_Risk 30.11.2023.xlsx'
#     xls = pd.ExcelFile(base_path + 'ML\\' + fn)
#     df = pd.DataFrame()
#     for sheet in xls.sheet_names:
#         print('\t', colored(sheet, 'light_cyan'), 'okunuyor')
#         dfs = pd.read_excel(xls, sheet_name=sheet)
#         df = pd.concat([df, dfs], ignore_index=True) if len(sheet_list) > 1 else dfs

#     df_gdlr = df.copy()
#     save_pickle(df_gdlr, 'df_gdlr')
#     xls.close()

# df_gdlr_backup = df_gdlr.copy()


	 1 okunuyor
	 2 okunuyor


In [225]:
# if os.path.exists(pickle_folder_path + 'df_gflr'):
#     df_gflr = load_pickle('df_gflr')
# else:
#     fn = 'Grup_Firmasız_Limit_Risk- 31.10.2023.xlsx'
#     xls = pd.ExcelFile(base_path + 'ML\\' + fn)
#     df = pd.DataFrame()
    # sheet_list = xls.sheet_names
#     for sheet in sheet_list:
#         print('\t', colored(sheet, 'light_cyan'), 'okunuyor')
#         dfs = pd.read_excel(xls, sheet_name=sheet)
#         df = pd.concat([df, dfs], ignore_index=True) if len(sheet_list) > 1 else dfs

#     df_gflr = df.copy()
#     save_pickle(df_gflr, 'df_gflr')
#     xls.close()

# df_gflr_backup = df_gflr.copy()


# fn = 'İSO İkinci 500.xlsx'
# xls = pd.ExcelFile(base_path + 'ML\\' + fn)
# df_iso500 = pd.read_excel(xls)


# fn = 'Tüm Firmalar Tüzel.csv'
# df_gflr_tuzel = pd.read_csv(base_path + 'ML\\' + fn, sep=';', encoding='ANSI')

In [26]:
fn = 'Tüzel Müşteri Listesi.csv'
df_tml = pd.read_csv(base_path + 'ML\\' + fn, sep=';', encoding='ANSI')

fn = 'İSO İkinci 500.xlsx'
xls = pd.ExcelFile(base_path + 'ML\\' + fn)
df_iso500 = pd.read_excel(xls)

In [237]:
# from thefuzz import fuzz

# dfm = df_gflr_backup[['Müşteri Numarası', 'Firma Unvanı']].copy()
# dfm = df_tml.copy()
# dfm.columns = ['Müşteri No', 'Adı']
# dfm = dfm.loc[dfm['Adı'].notnull()]
# dfm = dfm.drop_duplicates(keep='first')
# dfm = dfm.loc[dfm['Müşteri No'].isin(df_gflr_tuzel['Müşteri No'])]

def fix_string(x):
    sl = ['SANAYI', 'TICARET', 'TURIZM', 'DANISMANLIK', 'HIZMETLERI', 'ITHALAT', 'IHRACAT', 'IMALAT',
        'NAKLIYE', 'FABRIKASI', 'ENTEGRE']
    
    y = x.upper()
    y += ' '
    y = y.replace('İ', 'I').replace('Ü', 'U').replace('Ö', 'O').replace('Ç', 'C').replace('Ş', 'S').replace('Ğ', 'G')

    for c in ['AR-GE', 'AR GE ']:
        y = y.replace(c, 'ARASTIRMA GELISTIRME')

    for c in ['.']:
        y = y.replace(c, '. ')
        
    for c in [' VE ', '-', ',', '&', 'SINIRLI SORUMLU ']:
        y = y.replace(c, ' ')

    for c in sl:
        y = y.replace(c, ' ' + c + ' ')
    
    for c in [' SANAYI I ']:
        y = y.replace(c, ' SANAYI ')

    for c in ['T. A. S. ']:
        y = y.replace(c, 'TURK ')

    for c in ['AS.', 'A.S', 'A.S.', 'A. S.']:
        y = y.replace(c, '')
    
    for c in ['AS', 'A S', 'A. S', 'AS. ', 'A. S. ', 'A S']:
        if y.endswith(' ' + c):
            y = y[:-(len(c)+1)]
        elif y.endswith(' ' + c + ' '):
            y = y[:-(len(c)+2)]

    for c in ['SIRKETI', 'SIRKET', 'STI.', 'STI ', '\xa0']:
        y = y.replace(c, '')
        
    for c in ['A.']:
        y = y.replace(c, '')
        
    for c in ['LTD.', 'LTD ', 'LIM.', 'LIM ']:
        y = y.replace(c, '')

    for c in ['ANONIM', 'ANOMIN', 'LIMITED']:
        y = y.replace(c, '')

    for c in ['INS.', 'INS ']:
        y = y.replace(c, 'INSAAT')

    for c in ['SAN.', 'SAN ', 'SANAYII', 'SANVE ']:
        y = y.replace(c, 'SANAYI')

    for c in ['TIC.', 'TIC ', 'T.', ' VETIC', 'TICAR ']:
        y = y.replace(c, 'TICARET')

    for c in ['TUR.', 'TUR ']:
        y = y.replace(c, 'TURIZM')

    for c in ['DAN.', 'DAN ']:
        y = y.replace(c, 'DANSIMANLIK')

    for c in ['HIZ.', 'HIZ ', 'HIZM.', 'HIZM ']:
        y = y.replace(c, 'HIZMETLERI')

    for c in ['ITH.', 'ITH ']:
        y = y.replace(c, 'ITHALAT')

    for c in ['IHR.', 'IHR ']:
        y = y.replace(c, 'IHRACAT')

    for c in ['IML ', 'IMALATI']:
        y = y.replace(c, 'IMALAT')

    for c in ['NAK.', 'NAK ', 'NAKLIYECILIK']:
        y = y.replace(c, 'NAKLIYE')

    for c in ['GID.', 'GID ']:
        y = y.replace(c, 'GIDA')

    for c in ['ELK.']:
        y = y.replace(c, 'ELEKTRIK')

    for c in ['SIS.']:
        y = y.replace(c, 'SISTEMLERI')

    for c in ['MAD.', 'MAD ']:
        y = y.replace(c, 'MADDELERI')

    for c in ['MAM.', 'MAM ']:
        y = y.replace(c, 'MAMULLERI')

    for c in ['MAL.', 'MAL ']:
        y = y.replace(c, 'MALZEMELERI')

    for c in ['URUNLERIVE']:
        y = y.replace(c, 'URUNLERI')

    for c in ['FAB.', 'FAB ', 'FABRIKALARI']:
        y = y.replace(c, 'FABRIKASI')

    for c in sl:
        y = y.replace(c, ' ' + c + ' ')

    for c in [' VE ', '-', ',', '&', 'SINIRLI SORUMLU ']:
        y = y.replace(c, ' ')

    y = y.replace('.', ' ')

    for i in range(5):
        y = y.replace('  ', ' ')
    return y.strip()


In [281]:
fa = 'İSO500 Firma Adı'
ifa = 'İşlenmiş Firma Adı'
ma = 'Müşteri Adı'
mn = 'Müşteri No'
vkn = 'VKN'

dfm = df_tml.copy()
dfm.columns = [mn, ma]
dfm = dfm.loc[(dfm[ma].notnull()) & (dfm[mn].notnull())]
dfm = dfm.drop_duplicates(keep='first')
dfm[ifa] = dfm[ma].apply(fix_string)


dfi = df_iso500[['Kuruluş Adı']].copy()
dfi.columns = [fa]
dfi = dfi.loc[~dfi[fa].isin(['-', 'GENEL TOPLAM '])]
dfi[ifa] = dfi[fa].apply(fix_string)

dfif = pd.merge(dfi, dfm, on=ifa, how='inner')
dfif = dfif[[fa, ifa, ma, mn]]

dfie = dfi.loc[~dfi[fa].isin(dfif[fa])].copy().reset_index(drop=True)
dfie[vkn] = ''
dfie[mn] = ''

quick_export([dfif, dfie], 'İSOO500-2 Müşteri Listesi', ['Müşteriler', 'Müşteri Olmayanlar'])

In [327]:
file_name = 'İSO500 Müşteri Listesi.xlsx'
# file_name = 'İSOO500-2 Müşteri Listesi.xlsx'
xls = pd.ExcelFile(data_folder_path + file_name)
df = pd.DataFrame()
sheet_list = xls.sheet_names
for sheet in sheet_list:
    print('\t', colored(sheet, 'light_cyan'), 'okunuyor')
    dfs = pd.read_excel(xls, sheet_name=sheet)
    df = pd.concat([df, dfs], ignore_index=True) if len(sheet_list) > 1 else dfs
df_iso500_ml = df.copy()
xls.close()

	 Sayfa1 okunuyor


In [328]:
dfi = df_iso500_ml.copy()
dfx = pd.merge(dfi, df_kr202_last, on='Müşteri No', how='inner').drop_duplicates(subset='Müşteri No', keep='first')
l = len(dfx)
nx = dfx['Nakdi Reeskontlu Toplam'].sum() / 1_000_000
gx = dfx['G.Nakdi Toplam'].sum() / 1_000_000
tx = dfx['Toplam Reeskontlu Risk'].sum() / 1_000_000


df = pd.DataFrame(columns=['Küme', 'Adet', 'Nakdi Risk', 'G.Nakdi Risk', 'Toplam Risk'])
df.loc[len(df)] = ['İSO500 Tüm Firmalar', 500, '', '', '']
# df.loc[len(df)] = ['İSO İkinci 500 Tüm Firmalar', 500, '', '', '']
df.loc[len(df)] = ['Unvan Açıklayan Firmalar', len(dfi), '', '', '']
df.loc[len(df)] = ['Bankamızda Riski Olan Müşterilerimiz', l, nx, gx, tx]
df.loc[len(df)] = ['{} KR202, Reeskontlu, mio TL'.format(last_date.replace('.', '-')), '', '', '', '']

df_iso500_ozet = df.copy()
quick_export(df_iso500_ozet, 'İSO500 500 Özet')
# quick_export(df_iso500_ozet, 'İSO500-2 500 Özet')

In [333]:
dfie

,İSO500 Firma Adı,İşlenmiş Firma Adı,Müşteri No,VKN
0,Konya Kağıt San. ve Tic. A.Ş.,KONYA KAGIT SANAYI TICARET,NaN,NaN
1,Felda Iffco Gıda San. ve Tic. A.Ş.,FELDA IFFCO GIDA SANAYI TICARET,NaN,NaN
2,Sunar Özlem Gıda San. ve Tic. A.Ş.,SUNAR OZLEM GIDA SANAYI TICARET,NaN,NaN
3,Selkasan Kağıt ve Paketleme Malzemeleri İmalat...,SELKA SANAYI KAGIT PAKETLEME MALZEMELERI IMALA...,NaN,NaN
4,Uğurlular Tekstil San. ve Tic. A.Ş.,UGURLULAR TEKSTIL SANAYI TICARET,NaN,NaN
5,S.S. Marmara Zeytin Tarım Satış Kooperatifleri...,S S MARMARA ZEYTIN TARIM SATIS KOOPERATIFLERI ...,NaN,NaN
6,Master Builders Solutions Yapı Kimyasalları Sa...,MASTER BUILDERS SOLUTIONS YAPI KIMYASALLARI SA...,NaN,NaN
7,Turkpulse Dış Ticaret A.Ş.,TURKPULSE DIS TICARET,NaN,NaN
8,Groupe Atlantic İzmir Radyatör Sistemleri San....,GROUPE ATLAN TICARET IZMIR RADYATOR SISTEMLERI...,NaN,NaN
9,Staret Entegre Gıda San. ve Tic. A.Ş.,STARET ENTEGRE GIDA SANAYI TICARET,NaN,NaN


In [243]:
# gflr_banned = ['E', 'GIDA', 'ART', 'KAR', 'O KIMYA', 'E U']
# gflr_sektor = [
#     'ANONIM', 'SIRKETI', 'LIMITED', 'SANAYI', 'TICARET',
#     'TURIZM', 'DANISMANLIK', 'HIZMETLERI', 'METAL', 'ITHALAT', 'IHRACAT', 'IMALAT', 'NAKLIYECILIK',
#     'GIDA', 'KIMYA', 'ENERJI', 'PLASTIK', 'LASTIK', 'ELEKTRIK', 'OTOMOTIV', 'TEKSTIL', 'DOKUM',
#     'KAGIT', 'AMBALAJ', 'MENSUCAT', 'KABLO', 'URETIM', 'KIMYASAL', 'KIMYASALLARI', 'SERAMIK',
#     'CIMENTO', 'CAM', 'MALZEME', 'MALZEMELERI', 'MADDELERI', 'MAMUL', 'MAMULLERI', 'KARTON',
#     'MODA', 'TARIM', 'TARIMSAL', 'IC', 'DIS', 'GRANIT', 'YAPI', 'DENIM', 'HALI', 'ENTEGRE', 'GIYIM',
#     'INSAAT', 'MAKINA', 'ALUMINYUM', 'DEMIR', 'CELIK'
#     ]

# def check_sektor_only(df):
#     sl = df['X'].split(' ')
#     if len(sl) >= 1:
#         return all([x in gflr_sektor for x in sl])
#     return False

# dfm['XS'] = dfm.apply(check_sektor_only, axis=1)
# dfm = dfm.loc[dfm['XS'] == False]
# dfm = dfm.loc[~dfm['X'].isin(gflr_banned)]

In [244]:
# def fix_string_extra(x):
#     y = x
#     for c in [
#         'ANONIM', 'SIRKETI', 'LIMITED', 'SANAYI', 'TICARET',
#         'TURIZM', 'DANISMANLIK', 'HIZMETLERI', 'METAL', 'ITHALAT', 'IHRACAT', 'IMALAT', 'NAKLIYECILIK',
#         'GIDA', 'KIMYA', 'ENERJI', 'PLASTIK', 'LASTIK', 'ELEKTRIK', 'OTOMOTIV', 'TEKSTIL', 'DOKUM',
#         'KAGIT', 'AMBALAJ', 'MENSUCAT', 'KABLO', 'URETIM', 'KIMYASAL', 'KIMYASALLARI', 'SERAMIK',
#         'CIMENTO', 'CAM', 'MALZEME', 'MALZEMELERI', 'MADDELERI', 'MAMUL', 'MAMULLERI', 'KARTON',
#         'MODA', 'TARIM', 'TARIMSAL', 'IC', 'DIS', 'GRANIT', 'YAPI', 'DENIM', 'HALI', 'ENTEGRE', 'GIYIM',
#         'INSAAT', 'MAKINA', 'ALUMINYUM', 'DEMIR', 'CELIK',
#     ]:
#         y = y.replace(c + ' ', ' ')
#         if y.endswith(c):
#             y = y[:-len(c)]
#     for i in range(5):
#         y = y.replace('  ', ' ')
#     return y.strip()


In [245]:
# dfi['Y'] = dfi['X'].apply(fix_string_extra)
# dfm['Y'] = dfm['X'].apply(fix_string_extra)

In [246]:
# ys = []
# ms = []
# mns = []

# dfa = pd.DataFrame()
# for i, y in enumerate(dfi['Y']):
#     print(i)
#     dfm['Z'] = y
#     dfm['YZ'] = dfm.apply(lambda df: df['Z'] in df['Y'], axis=1)
#     dfa = v_stack([dfa, dfm.loc[dfm['YZ']]], 0)
#     # _ms = []
#     # _mns = []
#     # for m in dfm['Y']:
#     #     if i in m:
#     #         dft = dfm.loc[dfm['Y'] == m, ['Müşteri No', 'X']]
#     #         _mns.append(dft['Müşteri No'].values[0])
#     #         _ms.append(dft['X'].values[0])
#     # ys.append(y)
#     # mns.append(_mns)
#     # ms.append(_ms)

# dfii = dfi[['Kuruluş Adı', 'X', 'Y']]

# dfa.columns = ['Müşteri No', 'Adı', 'X', 'Y', 'Z', 'YZ']
# dfaa = dfa[['Z', 'X', 'Müşteri No']].rename(columns={'Z': 'Y'})

# dfz = pd.merge(dfii, dfaa, on='Y', how='left')
# dfz.columns = ['İSO500 Adı', 'İşlenmiş Ad', 'Anahtar', 'Müşteri Adı', 'Müşteri No']
# dfz = dfz.sort_values(['İSO500 Adı', 'Müşteri Adı'])

# quick_export(dfif, 'İSO500 İkinci 500 Otomatik')
# quick_export(dfz, 'İSO500 İkinci 500 Manuel')

In [247]:
# hsl = []
# hssl = []
# l = len(dfi)
# for i, name in enumerate(dfi['X']):
#     # hs = 0
#     # hss = ''
#     scores, matches = [], []
#     for match in dfm['X']:
#         score = fuzz.partial_ratio(name, match)
#         scores.append(score)
#         matches.append(match)
#     matches = [x for _, x in sorted(zip(scores, matches), reverse=True)]
#     scores = sorted(scores, reverse=True)
#     hsl.append(scores)
#     hssl.append(matches)
#     # print(i, f'{int((i + 1) / l * 100)}%')

In [82]:
# dfi.columns = ['İSO500 Firma Adı', 'İşlenmiş İSO500 Firma Adı']

# l = 20
# cl = ['M-' + str(x+1) for x in range(l)]
# mnl = ['MN-' + str(x+1) for x in range(l)]
# dfms = pd.DataFrame(columns = cl)

# for matches in hssl:
#     dfms.loc[len(dfms)] = matches[:l]

# dff = h_stack([dfi, dfms], 0)
# for c, mn in zip(cl, mnl):
#     dft = dfm[['Müşteri No', 'X']]
#     dft.columns = [mn, c]
#     dff = pd.merge(dff, dft, on=c, how='left')

# dff = dff.drop_duplicates(subset=dff.columns[0], keep='first')

# quick_export(dff, 'İSO İkinci 500 Run 7')